# Cell Volume vs Temperature from Rietveld Refinement

This notebook extracts unit cell volumes from a series of `.cif` files produced  
by Rietveld refinement (e.g. from TOPAS or FullProf), pairs each volume with  
a measured temperature from a log file, and plots the thermal expansion of the unit cell.

A linear fit extracts the volumetric thermal expansion coefficient:

$$\frac{dV}{dT} \approx a \quad [\text{Å}^3/\text{K}]$$

**Directory structure expected:**

```
data/
  cif/
    001_refinement.cif
    002_refinement.cif
    ...
  refinement.log
```

The log file must have three whitespace-separated columns per line:  
`trial_number   <anything>   temperature_K`


## 1. Imports

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from scipy.optimize import curve_fit


## 2. Plot Formatting Helpers


In [ ]:
def format_my_plot(figsize=(6, 6)):
    """Publication-style axes: inward ticks on all sides, minor ticks."""
    plt.gcf().set_size_inches(*figsize)
    plt.rcParams.update({'axes.linewidth': 1.2, 'font.size': 14})
    plt.locator_params(axis='y', nbins=6)
    plt.tick_params(axis='y', left=True, right=True, direction='in', length=13, width=1.2)
    plt.tick_params(axis='x', bottom=True, top=True, direction='in', length=13, width=1.2)
    plt.gca().yaxis.set_minor_locator(AutoMinorLocator(2))
    plt.gca().xaxis.set_minor_locator(AutoMinorLocator(2))
    plt.tick_params(which='minor', direction='in', left=True, right=True,
                    bottom=True, top=True, length=7, width=1.2)
    plt.gca().set_aspect(1.0 / plt.gca().get_data_ratio())
    plt.tight_layout()

def format_legend():
    leg = plt.legend(framealpha=1)
    leg.get_frame().set_edgecolor('black')
    leg.get_frame().set_facecolor('white')
    leg.get_frame().set_linewidth(1.1)


## 3. Load Data

Parse `.cif` files for `_cell_volume` and pair with temperatures from the log file.


In [ ]:
# ── User inputs ───────────────────────────────────────────────
CIF_DIR  = 'data/cif'            # folder containing numbered .cif files
LOG_FILE = 'data/refinement.log' # log file: trial_number  <col>  temperature_K
# ─────────────────────────────────────────────────────────────

cell_volume_pattern = re.compile(r'_cell_volume\s+([\d.]+)')

# Extract cell volumes from .cif files
# File names must start with an integer trial number, e.g. "001_sample.cif"
cell_volumes = {}
for fname in os.listdir(CIF_DIR):
    if fname.endswith('.cif'):
        try:
            trial = int(fname.split('_')[0])
        except ValueError:
            continue
        with open(os.path.join(CIF_DIR, fname)) as f:
            match = cell_volume_pattern.search(f.read())
        if match:
            cell_volumes[trial] = float(match.group(1))

# Extract temperatures from log file
temperatures = {}
with open(LOG_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 3:
            try:
                temperatures[int(parts[0])] = float(parts[2])
            except ValueError:
                continue

# Merge into a DataFrame
records = [
    {'Trial': t, 'Cell Volume (Å³)': cell_volumes[t], 'Temperature (K)': temperatures[t]}
    for t in cell_volumes if t in temperatures
]
df = pd.DataFrame(records).sort_values('Temperature (K)').reset_index(drop=True)

# Normalize cell volume relative to the highest-temperature measurement
df['Normalized Volume'] = df['Cell Volume (Å³)'] / df['Cell Volume (Å³)'].max()

print(df.to_string(index=False))


## 4. Linear Fit

Fit a line to $V(T)$ to extract the mean volumetric thermal expansion rate $dV/dT$.


In [ ]:
def linear(T, a, b):
    return a * T + b

T_data = df['Temperature (K)'].values
V_data = df['Cell Volume (Å³)'].values

popt, pcov = curve_fit(linear, T_data, V_data)
a, b = popt
a_err, b_err = np.sqrt(np.diag(pcov))

print(f'dV/dT  = {a:.4f} ± {a_err:.4f}  Å³/K')
print(f'V(0 K) = {b:.4f} ± {b_err:.4f}  Å³  (extrapolated intercept)')

V_fit = linear(T_data, *popt)


## 5. Plots

### 5a. Cell Volume vs Temperature with Linear Fit


In [ ]:
plt.figure()
plt.scatter(T_data, V_data, color='steelblue', zorder=5, label='Rietveld data')
plt.plot(T_data, V_fit, color='firebrick', linestyle='--',
         label=f'Fit: $dV/dT = {a:.3f}$ Å³/K')
plt.xlabel('Temperature (K)')
plt.ylabel('Cell Volume (Å³)')
plt.title('Unit Cell Volume vs Temperature')
format_my_plot(figsize=(9, 6))
format_legend()
plt.tight_layout()
plt.show()


### 5b. Normalized Cell Volume

Normalized to the room-temperature (highest-T) measurement.  
Useful for comparing thermal contraction across different samples or compounds.


In [ ]:
plt.figure()
plt.plot(df['Temperature (K)'], df['Normalized Volume'],
         marker='o', linestyle='-', color='steelblue')
plt.xlabel('Temperature (K)')
plt.ylabel('Normalized Volume $V / V_{T_{\\max}}$')
plt.title('Normalized Unit Cell Volume vs Temperature')
format_my_plot(figsize=(9, 6))
plt.tight_layout()
plt.show()
